In [6]:
from pathlib import Path
import sys
import os
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
LABELS_PATH = DATA_DIR / "labels.csv"
MODEL_PATH = MODELS_DIR / "best_model.pkl"
METRICS_PATH = RESULTS_DIR / "metrics.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
from src.classifier import load_labeled_data, train_and_save_model

labels_df = load_labeled_data(str(LABELS_PATH))
labels_df.head()

,candidate_id,page_stub,page_url,image_url,local_path,domain,file_name,width,height,format,...,file_size_bytes,area,aspect_ratio,is_downloaded_and_valid,label,split,is_tiny,is_suspicious_domain,has_ui_keyword,target
0,page_01_cand_000000,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/105674087,data/raw/page_01/page_01_cand_000000_05b4d2d19...,mc.yandex.ru,105674087,1.0,1.0,GIF,...,43.0,1.0,1.0,True,non_content,val,True,True,False,0.0
1,page_01_cand_000001,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/27509004,data/raw/page_01/page_01_cand_000001_105a14e1b...,mc.yandex.ru,27509004,1.0,1.0,GIF,...,43.0,1.0,1.0,True,non_content,val,True,True,False,0.0
2,page_01_cand_000002,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://mc.yandex.ru/watch/26649402,data/raw/page_01/page_01_cand_000002_9e463b829...,mc.yandex.ru,26649402,1.0,1.0,GIF,...,43.0,1.0,1.0,True,non_content,val,True,True,False,0.0
3,page_01_cand_000003,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://counter.rambler.ru/top100.cnt?pid=2635991,data/raw/page_01/page_01_cand_000003_1ec34c737...,counter.rambler.ru,top100.cnt,1.0,1.0,GIF,...,43.0,1.0,1.0,True,non_content,val,True,True,True,0.0
4,page_01_cand_000004,page_01,https://eda.rambler.ru/media/recepty/pyat-horo...,https://counter.rambler.ru/top100.cnt?pid=7728281,data/raw/page_01/page_01_cand_000004_5ffb4dcfb...,counter.rambler.ru,top100.cnt,1.0,1.0,GIF,...,43.0,1.0,1.0,True,non_content,val,True,True,True,0.0


In [7]:
labels_df['label'].value_counts(), labels_df['split'].value_counts()

(label
 content        363
 non_content    288
 Name: count, dtype: int64,
 split
 train    387
 val      137
 test     127
 Name: count, dtype: int64)

In [8]:
report = train_and_save_model(
    labels_csv_path=str(LABELS_PATH),
    model_path=str(MODEL_PATH),
    model_type='logreg',
)
report['threshold']

0.68

In [9]:
metrics_df = pd.DataFrame([
    {'split': 'train', **report['train_metrics']},
    {'split': 'val', **report['val_metrics']},
    {'split': 'test', **report['test_metrics']},
])
metrics_df[['split', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn', 'tn']]

,split,precision,recall,f1,tp,fp,fn,tn
0,train,0.952381,0.176991,0.298507,40,2,186,159
1,val,0.875000,0.205882,0.333333,14,2,54,67
2,test,1.000000,0.159420,0.275000,11,0,58,58


In [10]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(METRICS_PATH, index=False)
metrics_df

,split,precision,recall,f1,accuracy,tp,fp,fn,tn,n,mean_proba,median_proba,proba_p90
0,train,0.952381,0.176991,0.298507,0.514212,40,2,186,159,387,NaN,NaN,NaN
1,val,0.875000,0.205882,0.333333,0.591241,14,2,54,67,137,0.479699,0.522397,0.705102
2,test,1.000000,0.159420,0.275000,0.543307,11,0,58,58,127,0.530494,0.554622,0.672511
